# Examine chat templates and system-prompt format

For each candidate LLM, document the chat template and
system-prompt format so the prompts the search pipeline
builds are correct by construction (M1, llm-prm-deep-dive).

Two template paths matter, and they differ:

1. **Native** — the template shipped in each model's
   `tokenizer_config.json`. Llama 3.2 uses
   `<|start_header_id|>role<|end_header_id|>` + `<|eot_id|>`;
   Qwen 2.5 uses `<|im_start|>role ... <|im_end|>`.
2. **Pipeline** — `sal.Config.custom_chat_template`, a single
   hardcoded Llama-3.1 template applied to *every* model when
   `config.custom_chat_template is not None`. Applied to a Qwen
   tokenizer it overrides Qwen's native format with Llama-style
   headers. This notebook renders both so the override is
   visible, not silent.

The conversation is built with `sal.search.utils.build_conv`
(system + user + optional assistant), matching what BoN / MCTS
search actually send.

**Separator check.** Reasoning steps are joined with `\n\n`.
When the assistant turn ends with `\n\n` and the prompt is
rendered with `continue_final_message=True`,
`apply_chat_template` can silently trim or crash on the
trailing separator (see `docs/findings.md` 2026-06-12). For
each model and each template path, this notebook checks whether
the trailing `\n\n` survives templating.

**GPU:** not required — tokenizer-only, runs in seconds.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append("..")

from transformers import AutoTokenizer

from utils.configs import GenConfig, build_conv

In [ ]:
base_dir = '/groups/chichengz/tnn/datasets/'

model_names = [
    "Llama3.2-1B-Instruct",
    "Llama3.2-3B-Instruct",
    "Qwen2.5-3B-Instruct",
    "Qwen2.5-7B-Instruct",
]

# Pull the exact system prompt and custom template the search
# pipeline uses, so we examine the real prompt path.
config = GenConfig()
config.date_string = "Aug 1 2025"

# A short sample problem and a two-step assistant response that
# ends with the \n\n step separator (the case that can break).
sample_question = "What is 2 + 2?"
sample_response = (
    "## Step 1: Add the numbers\n"
    "2 + 2 = 4.\n\n"
    "## Step 2: State the answer\n"
    "Therefore, the final answer is: $\\boxed{4}$.\n\n"
)

## Helpers

`render` builds the conversation and applies a template along
the same call path as search: `continue_final_message=True`
when there is an assistant turn, so the prompt is left open for
the model to continue. `sep_survives` checks whether a trailing
`\n\n` in the assistant content is still present after
rendering.

In [ ]:
def render(tokenizer, question, response, system_prompt,
           custom_template=None):
    """Render a built conversation, mirroring the search path.

    If custom_template is given it overrides the tokenizer's
    native chat_template (what the pipeline does when
    config.custom_chat_template is set). Returns the rendered
    string, or the exception repr if templating raises.
    """
    convs = [build_conv(question, response, system_prompt)]
    if custom_template is not None:
        tokenizer.chat_template = custom_template
    has_assistant = response != ""
    try:
        out = tokenizer.apply_chat_template(
            convs,
            add_generation_prompt=not has_assistant,
            continue_final_message=has_assistant,
            tokenize=False,
        )
        return out[0]
    except Exception as e:
        return f"<RAISED: {type(e).__name__}: {e}>"


def sep_survives(rendered, response):
    """True if a trailing \\n\\n in `response` is still present at
    the tail of the rendered prompt."""
    if not response.endswith("\n\n"):
        return None
    if rendered.startswith("<RAISED:"):
        return False
    return rendered.endswith("\n\n")

## System prompt

Single shared system prompt from `Config` — same for every
model. Printed once below.

In [ ]:
print("=== Shared system prompt (Config.system_prompt) ===")
print(config.system_prompt)

## Native templates

For each model, render with its own shipped chat template (no
override). Inspect the turn markers and whether the trailing
step separator survives.

In [ ]:
native_sep = {}

for name in model_names:
    print(f"\n{'=' * 60}\n=== {name} (native template) ===\n{'=' * 60}")
    tok = AutoTokenizer.from_pretrained(
        base_dir + name, trust_remote_code=True,
    )
    rendered = render(
        tok, sample_question, sample_response,
        config.system_prompt, custom_template=None,
    )
    print(rendered)
    survived = sep_survives(rendered, sample_response)
    native_sep[name] = survived
    print(f"\n  trailing '\\n\\n' survives: {survived}")

## Pipeline template (`Config.custom_chat_template`)

The single hardcoded template applied to every model by the
search code. Watch the Qwen models in particular: their native
`<|im_start|>` format is replaced by Llama-style headers here.

In [ ]:
custom_sep = {}

for name in model_names:
    print(f"\n{'=' * 60}\n=== {name} (custom_chat_template) ===\n{'=' * 60}")
    tok = AutoTokenizer.from_pretrained(
        base_dir + name, trust_remote_code=True,
    )
    rendered = render(
        tok, sample_question, sample_response,
        config.system_prompt,
        custom_template=config.custom_chat_template,
    )
    print(rendered)
    survived = sep_survives(rendered, sample_response)
    custom_sep[name] = survived
    print(f"\n  trailing '\\n\\n' survives: {survived}")

## Summary: separator survival

`True` = trailing `\n\n` preserved; `False` = trimmed or
templating raised. If native and pipeline disagree, the
pipeline path (custom template) is the one that matters, since
that is what search actually sends.

In [ ]:
print(f"{'model':<24}{'native':>10}{'pipeline':>12}")
print('-' * 46)
for name in model_names:
    print(
        f"{name:<24}{str(native_sep[name]):>10}"
        f"{str(custom_sep[name]):>12}"
    )